# 🚪 controller_explore — the bouncer's judgment (M4.1 →)

Everything the controller ever decides is in this notebook's two objects: a **pure
function** (the predicate — six checks, first failure wins) and a **dict** (the session
state machine). No LLM, no I/O, no side effects — you can hold the entire authorization
logic of the system in your head, and this notebook exists so you literally do.

Grows with Phase 4: §1–§3 predicate + machine (M4.1) · auth (M4.2) · translators (M4.3) ·
the HTTP API (M4.4) land as new sections.

**Companions:** [`docs/05-controller-spec.md`](../../docs/05-controller-spec.md) (the
rulebook) · `controller/src/controller/domain.py` (≈100 lines, half comments) ·
`controller/tests/test_domain.py` (every cell here, pinned).

## 1. The predicate: six boring checks, in a contract order

`CANONICAL_ENTITLEMENT_VIEW` is ticket #7 exactly as the controller reads it off the
chain. At 14:02, with Ada asking, everything passes:

In [1]:
from a2a_interfaces import ErrorCode
from a2a_interfaces.fixtures import ADA, BELL, CANONICAL_ENTITLEMENT_VIEW, WINDOW
from controller.domain import predicate, transition, IllegalTransition, TRANSITIONS

VIEW = CANONICAL_ENTITLEMENT_VIEW
verdict = predicate(VIEW, owner=ADA, requester=ADA, now=WINDOW.start + 120, active_ids=set())
print("14:02, Ada asks →", verdict, "  (None = come in)")


14:02, Ada asks → None   (None = come in)


## 2. Flip one fact at a time, watch the named refusal

Each denial is an `ErrorCode` from the shared enum (docs/03 §3.4) — the same strings the
HTTP API will return (M4.4) and the agents will read (M5.x):

In [2]:
IN_WINDOW = WINDOW.start + 120

cases = {
    "Bell asks instead of Ada":  dict(view=VIEW, owner=ADA, requester=BELL, now=IN_WINDOW),
    "Ada asks at 13:00":         dict(view=VIEW, owner=ADA, requester=ADA, now=WINDOW.start - 3600),
    "Ada asks at 16:00 sharp":   dict(view=VIEW, owner=ADA, requester=ADA, now=WINDOW.end),
    "ticket was revoked":        dict(view=VIEW.model_copy(update={"revoked": True}), owner=ADA, requester=ADA, now=IN_WINDOW),
    "serviceType nobody knows":  dict(view=VIEW.model_copy(update={"service_type": 7}), owner=ADA, requester=ADA, now=IN_WINDOW),
}
for story, kw in cases.items():
    print(f"{story:28} → {predicate(active_ids=set(), **kw)}")

print(f"{'#7 already has a session':28} → {predicate(VIEW, ADA, ADA, IN_WINDOW, active_ids={7})}")


Bell asks instead of Ada     → ErrorCode.E_NOT_OWNER
Ada asks at 13:00            → ErrorCode.E_NOT_STARTED
Ada asks at 16:00 sharp      → ErrorCode.E_EXPIRED
ticket was revoked           → ErrorCode.E_REVOKED
serviceType nobody knows     → ErrorCode.E_SCOPE
#7 already has a session     → ErrorCode.E_CONFLICT


**The order is contract, not accident.** A request that fails *everything* still
answers `E_NOT_OWNER` — first check, first answer. Try to predict, then run:

In [3]:
wreck = VIEW.model_copy(update={"revoked": True, "service_type": 9})
print(predicate(wreck, owner=ADA, requester=BELL, now=WINDOW.end + 999, active_ids={7}))


ErrorCode.E_NOT_OWNER


## 3. The state machine is a dict you can read

Not `if`-chains — data. Legal moves are keys; terminal states absorb `teardown`
(rule 8: tearing down what's already down is a success); everything else raises
`IllegalTransition`, which means a *programming* error, never a user denial:

In [4]:
from a2a_interfaces import SessionState

for (state, event), nxt in TRANSITIONS.items():
    print(f"{state.value:11} --{event:17}→ {nxt.value}")

print()
print("torn_down --teardown→", transition(SessionState.TORN_DOWN, "teardown").value, " (absorbed)")
try:
    transition(SessionState.REQUESTED, "provision_ok")   # skipped authorization!
except IllegalTransition as err:
    print("requested --provision_ok→ raises:", err)


requested   --authorize        → authorized
requested   --deny             → failed
authorized  --provision_ok     → active
authorized  --provision_failed → failed
active      --teardown         → torn_down

torn_down --teardown→ torn_down  (absorbed)
requested --provision_ok→ raises: no transition for event 'provision_ok' in state 'requested'


## 4. Rule 4, checked with your own eyes

The domain imports nothing that can do I/O — run the same inspection the test suite
pins (`test_domain_purity_no_io_imports`):

In [5]:
import ast, inspect
from controller import domain

tree = ast.parse(inspect.getsource(domain))
imports = sorted({
    (a.name if isinstance(n, ast.Import) else n.module).split(".")[0]
    for n in ast.walk(tree)
    if isinstance(n, (ast.Import, ast.ImportFrom))
    for a in (n.names if isinstance(n, ast.Import) else [n])
    if (a.name if isinstance(n, ast.Import) else n.module)
})
print("domain.py imports:", imports)
assert set(imports) <= {"__future__", "a2a_interfaces"}
print("no web3, no pygnmi, no HTTP, no filesystem — the judgment is portable ✓")


domain.py imports: ['__future__', 'a2a_interfaces']
no web3, no pygnmi, no HTTP, no filesystem — the judgment is portable ✓


## Experiments to try

- The window check: find the exact second Ada's ticket dies (hint: §2's third case).
  Why `now >= end_time` and not `>`? What would one extra second cost the provider?
- Give the predicate `known_service_types=(0,)` and a telemetry view — this is exactly
  how the M0.3 stub scopes itself to what it can translate.
- Draw `TRANSITIONS` as a diagram from the dict alone; compare with docs/05 §3.
- Break it: in `domain.py`, swap the order of the revoked and expired checks, run
  `uv run pytest controller/` — which named test catches it?